# Subscription-init / Health Locker access test

Tests spec §8.3.2 (`POST /subscription-requests/v3/init`) — the mechanism §8.1 describes as: *"Subscription will get auto approve for health locker for all HIPs and for all HI types."* Getting this call ACCEPTED is the actual goal (ongoing Health Locker access), not just a 202.

**Standalone — no import from `server.auth`/`server.hiu_consent` etc.** Mirrors `aegle-phr/aegle_phr/phr/subscription.py`'s own `initiate_subscription_request()` request shape (URL, headers, payload) directly, so this notebook can mint a token for whichever identity is under test (`server.config.CLIENT_ID` or `server.config.PHR_CLIENT_ID`) without being tied to whichever one `aegle_phr`'s own settings/gateway-token cache happens to be configured for.

## Findings so far (2026-09-07) — read before re-running

Every combination tried gets the identical `{"code": "ABDM-1040: ", "message": "Invalid HIU ID"}`, across **two separate bridge registrations**:

- `SBXID_046112` (the original bridge, `server.config.CLIENT_ID`) as `hiu.id` — rejected.
- `IN3310002215`, `IN3310002220`, `IN2410002587`, `IN2410002590` (the 4 registered facility ids) as `hiu.id`, under `SBXID_046112`'s own token — all rejected.
- `SBXID_073333` (the new bridge, `server.config.PHR_CLIENT_ID`) as `hiu.id`, using **its own** token (identity matched to token) — rejected, identically.
- All 4 facility ids again, this time under `SBXID_073333`'s own token — all rejected, identically.
- `purpose.code = "PATRQT"` ("Self Requested") instead of `"CAREMGT"` — no difference, same rejection.

**One narrowing data point**: spec §8.3.6 (`.../hiu/on-notify`, the ack endpoint — needs no `hiu.id` and no patient session, just the gateway bearer token) does **NOT** reject `SBXID_073333` with "Invalid HIU ID" — it accepts the call far enough to instead complain `{"code": "ABDM-1015", "message": "Invalid Response"}` about a deliberately made-up correlation id. So this identity **is** authorized to reach Subscription-flow endpoints at the gateway level — the rejection is specifically ABDM's registry not recognizing any of the 5 known strings as a valid `hiu.id` value for `init`, not a blanket block on the whole registration.

See the `project-hiu-role-not-provisioned` memory for the full history. Real next step is almost certainly ABDM/NHA support, not another payload variant — but this notebook is here so a new identity, once one exists, or a hint from ABDM support about what value they actually expect, can be tried in under a minute.

## Setup

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\hp\Desktop\Aayush\repo")

import json
from datetime import datetime, timedelta, timezone

import requests

from server.config import CLIENT_ID, CLIENT_SECRET, GATEWAY_BASE_URL, HIECM_BASE_URL, HIPS, PHR_CLIENT_ID, PHR_CLIENT_SECRET, X_CM_ID
from server.utils import generate_request_id, generate_timestamp

print("CLIENT_ID (old bridge):", CLIENT_ID)
print("PHR_CLIENT_ID (new bridge) set:", bool(PHR_CLIENT_ID))

## Which identity to test

Set `TEST_CLIENT_ID`/`TEST_CLIENT_SECRET` to whichever registration you want to run against — defaults to the new one (`PHR_CLIENT_ID`). Swap to `CLIENT_ID`/`CLIENT_SECRET` to re-run against the old bridge instead.

In [ ]:
TEST_CLIENT_ID = PHR_CLIENT_ID      # <-- swap to CLIENT_ID to test the old bridge instead
TEST_CLIENT_SECRET = PHR_CLIENT_SECRET  # <-- swap to CLIENT_SECRET to match

PATIENT_ABHA_ADDRESS = "poojaanchaliya@sbx"  # <-- change if testing against a different patient

if not TEST_CLIENT_ID or not TEST_CLIENT_SECRET:
    raise SystemExit("TEST_CLIENT_ID / TEST_CLIENT_SECRET not set -- fill in repo/.env first.")

## Step 1 — Gateway token

Mints a fresh token AS `TEST_CLIENT_ID` — not cached, not shared with `server.utils.get_gateway_token()` (which is wired to `CLIENT_ID`/`CLIENT_SECRET` specifically, for the live app). Re-run this cell to get a fresh token if it's been >20 minutes (`expiresIn` below).

In [ ]:
token_response = requests.post(
    url=f"{GATEWAY_BASE_URL}/sessions",
    json={"clientId": TEST_CLIENT_ID, "clientSecret": TEST_CLIENT_SECRET, "grantType": "client_credentials"},
    headers={
        "Content-Type": "application/json",
        "REQUEST-ID": generate_request_id(),
        "TIMESTAMP": generate_timestamp(),
        "X-CM-ID": X_CM_ID,
    },
    timeout=30,
)
print("Token generation status:", token_response.status_code)
if token_response.status_code != 200:
    print("Body:", token_response.text)
    raise SystemExit("Token generation failed -- fix this before continuing.")

_token_data = token_response.json()
access_token = _token_data["accessToken"]
print("expiresIn (seconds):", _token_data.get("expiresIn"))
# Deliberately not printing access_token itself -- it's a live bearer credential.

## Step 2 — Request helper

In [ ]:
def subscription_init(hiu_id: str, purpose_code: str = "CAREMGT", purpose_text: str = "Care Management") -> requests.Response:
    now = datetime.now(timezone.utc)
    period_from = (now + timedelta(minutes=10)).isoformat(timespec="milliseconds").replace("+00:00", "Z")
    period_to = (now + timedelta(days=36500)).isoformat(timespec="milliseconds").replace("+00:00", "Z")
    payload = {
        "subscription": {
            "purpose": {"text": purpose_text, "code": purpose_code, "refUri": "www.abdm.gov.in"},
            "patient": {"id": PATIENT_ABHA_ADDRESS},
            "hiu": {"id": hiu_id},
            "categories": ["LINK", "DATA"],
            "period": {"from": period_from, "to": period_to},
        }
    }
    headers = {
        "Content-Type": "application/json",
        "REQUEST-ID": generate_request_id(),
        "TIMESTAMP": generate_timestamp(),
        "Authorization": f"Bearer {access_token}",
        "X-CM-ID": X_CM_ID,
    }
    url = f"{HIECM_BASE_URL}/subscription-requests/v3/init"
    response = requests.post(url, json=payload, headers=headers, timeout=30)
    print(f"hiu.id={hiu_id!r:16} purpose={purpose_code:8} -> status={response.status_code}", end=" ")
    try:
        print("body=", json.dumps(response.json()))
    except ValueError:
        print("body(non-JSON)=", response.text)
    return response

## Step 3 — Try it: self-subscription (`hiu.id = TEST_CLIENT_ID`)

This is the actual test — if this ever returns 202, subscription-init succeeded and the follow-up flow (get-all-requests, approve, lockers — see `aegle-phr/notebooks/p13_subscription_flow_test.ipynb` for the rest of spec §8 once there's something real to work with) becomes worth running.

In [ ]:
init_response = subscription_init(hiu_id=TEST_CLIENT_ID)

## Step 4 — Sweep every known identity, for convenience

Re-runs the exact combinations already confirmed rejected (see the summary at the top) — useful to re-check quickly if ABDM support says something changed, without hand-editing `hiu.id` five times.

In [ ]:
candidates = [TEST_CLIENT_ID] + [hip["hip_id"] for hip in HIPS]
sweep_results = {hiu_id: subscription_init(hiu_id=hiu_id) for hiu_id in candidates}

## Step 5 — If anything above returned 202: check status

Patient-facing (`GET /subscription-requests/v3/requests`) — needs a real patient `X-AUTH-TOKEN`, not just the gateway token above. Paste one in below if Step 3/4 ever succeeds and you want to see whether the request resolved GRANTED/DENIED or is still sitting REQUESTED.

In [ ]:
X_AUTH_TOKEN = ""  # <-- paste a real patient session token here, only needed if Step 3/4 got a 202

if X_AUTH_TOKEN:
    status_response = requests.get(
        url=f"{HIECM_BASE_URL}/subscription-requests/v3/requests",
        params={"limit": 10, "offset": 0, "status": "ALL"},
        headers={
            "REQUEST-ID": generate_request_id(),
            "TIMESTAMP": generate_timestamp(),
            "Authorization": f"Bearer {access_token}",
            "X-AUTH-TOKEN": X_AUTH_TOKEN,
            "X-CM-ID": X_CM_ID,
        },
        timeout=30,
    )
    print("status:", status_response.status_code)
    print("body:", json.dumps(status_response.json(), indent=2))
else:
    print("No X_AUTH_TOKEN set -- skipping (nothing to check unless Step 3/4 returned a 202).")

## Reference — the one non-`hiu.id` probe (spec §8.3.6)

Already run once (see the findings summary at the top) — kept here so it's reproducible, not just reported. No `hiu.id`, no patient session; just the gateway bearer token plus a made-up correlation id (there's no real one to use, since `init` never succeeds). The point isn't a successful ack — it's *which* error comes back.

In [ ]:
import uuid

ack_response = requests.post(
    url=f"{HIECM_BASE_URL}/subscription-requests/v3/hiu/on-notify",
    json={
        "acknowledgement": {"status": "OK", "subscriptionRequestId": str(uuid.uuid4())},
        "response": {"requestId": str(uuid.uuid4())},
    },
    headers={
        "Content-Type": "application/json",
        "REQUEST-ID": generate_request_id(),
        "TIMESTAMP": generate_timestamp(),
        "Authorization": f"Bearer {access_token}",
        "X-CM-ID": X_CM_ID,
    },
    timeout=30,
)
print("status:", ack_response.status_code)
print("body:", json.dumps(ack_response.json(), indent=2))